# BeforeYouAgree Risk Classification

This notebook implements the Iteration 1 baseline for classifying Terms of Service and Privacy Policy clauses as **Low**, **Medium**, or **High** risk.

The workflow is intentionally simple and explainable: **CountVectorizer → TF-IDF → Logistic Regression**. The text is converted into numerical word features, several Logistic Regression settings are compared on the validation set, and the selected model is evaluated once on the held-out test set.

The input files use automatically generated provisional labels. Therefore, the reported metrics show how well the model reproduces the current labelling rules; they do not prove legal correctness.

In [1]:
# Standard-library tools for file paths, JSON output, and model serialization.
from pathlib import Path
import json
import pickle
# NumPy and pandas are used for numerical calculations and tabular data.
import numpy as np
import pandas as pd

# Scikit-learn provides the text feature extraction, classifier, and metrics.
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, classification_report
)

## 1. Load and validate the cleaned OTA data

The three datasets were separated by service rather than by random clause rows. This reduces the chance that very similar clauses from the same service appear in both training and testing data. The balanced files retain every available Medium- and High-risk example while reducing the much larger Low-risk class.

In [2]:
# Keep input and output locations in one place so the notebook is easy to move.
DATA_DIR = Path('outputs/ota_weak_labelled_v1')
OUTPUT_DIR = Path('outputs/ota_machine_learning_risk_model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def load_data(filename):
    """Read one split and stop immediately if an unexpected label is found."""
    data = pd.read_csv(DATA_DIR / filename)
    valid_levels = {'Low', 'Medium', 'High'}
    if not set(data['risk_level'].dropna()).issubset(valid_levels):
        raise ValueError('Unexpected risk label found')
    return data

# Training data fits the model, validation data selects settings, and test
# data is reserved for the final evaluation.
train_data = load_data('ota_train_balanced.csv')
validation_data = load_data('ota_val_balanced.csv')
test_data = load_data('ota_test_balanced.csv')

print('Train:', train_data['risk_level'].value_counts().to_dict())
print('Validation:', validation_data['risk_level'].value_counts().to_dict())
print('Test:', test_data['risk_level'].value_counts().to_dict())

Train: {'Low': 240, 'Medium': 188, 'High': 48}
Validation: {'Low': 40, 'Medium': 33, 'High': 6}
Test: {'Low': 45, 'Medium': 36, 'High': 9}


## 2. Convert clause text into TF-IDF features

`CountVectorizer` builds a vocabulary from the training clauses and counts individual words and two-word phrases. `TfidfTransformer` then gives more importance to terms that are informative in a particular clause and less importance to terms that occur throughout the dataset.

The vocabulary and TF-IDF weights are fitted on the training set only. Validation and test data are transformed with the already-fitted objects to prevent information leakage.

In [3]:
# Use unigrams and bigrams so the model can learn both individual words
# (for example, 'arbitration') and short phrases ('automatic renewal').
count_vectorizer = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Fit the vocabulary only on training text. Calling transform on validation
# and test text prevents those datasets from influencing feature creation.
X_train_count = count_vectorizer.fit_transform(train_data['clause_text_clean'])
X_validation_count = count_vectorizer.transform(validation_data['clause_text_clean'])
X_test_count = count_vectorizer.transform(test_data['clause_text_clean'])

# sublinear_tf replaces a raw count with 1 + log(count), which prevents a
# frequently repeated word inside one long clause from dominating the model.
tfidf_transformer = TfidfTransformer(sublinear_tf=True)
X_train = tfidf_transformer.fit_transform(X_train_count)
X_validation = tfidf_transformer.transform(X_validation_count)
X_test = tfidf_transformer.transform(X_test_count)

# Separate the target labels from the feature matrices.
y_train = train_data['risk_level']
y_validation = validation_data['risk_level']
y_test = test_data['risk_level']

print('Training feature shape:', X_train.shape)

Training feature shape: (476, 5272)


## 3. Select Logistic Regression settings using validation data

Logistic Regression is a suitable baseline because it is fast, works well with sparse TF-IDF features, and produces probabilities for all three classes.

`C` controls regularization: a smaller value applies stronger regularization, while a larger value lets the model fit the training data more closely. `class_weight='balanced'` gives more importance to classes with fewer training examples.

The settings are ranked primarily by validation Macro-F1 because Macro-F1 gives Low, Medium, and High equal importance. Validation accuracy is used only as a secondary tie-breaker.

In [4]:
# Compare a small, transparent set of regularization and class-weight options.
candidate_settings = [
    {'C': 0.1, 'class_weight': None},
    {'C': 1.0, 'class_weight': None},
    {'C': 10.0, 'class_weight': None},
    {'C': 0.1, 'class_weight': 'balanced'},
    {'C': 1.0, 'class_weight': 'balanced'},
    {'C': 10.0, 'class_weight': 'balanced'}
]

validation_results = []

# Fit each candidate on exactly the same training features and assess it on
# the validation set. The test set is not used during model selection.
for setting in candidate_settings:
    model = LogisticRegression(
        C=setting['C'],
        class_weight=setting['class_weight'],
        max_iter=2000,
        random_state=42
    )

    model.fit(X_train, y_train)
    validation_prediction = model.predict(X_validation)

    validation_results.append({
        'C': setting['C'],
        'class_weight': setting['class_weight'],
        'validation_accuracy': accuracy_score(y_validation, validation_prediction),
        'validation_macro_f1': f1_score(
            y_validation, validation_prediction, average='macro'
        )
    })

# Place the strongest validation result in the first row.
validation_table = pd.DataFrame(validation_results).sort_values(
    ['validation_macro_f1', 'validation_accuracy'],
    ascending=False
).reset_index(drop=True)

validation_table

,C,class_weight,validation_accuracy,validation_macro_f1
0,10.0,balanced,0.696203,0.621142
1,1.0,balanced,0.683544,0.613238
2,10.0,None,0.658228,0.523884
3,0.1,balanced,0.645570,0.519424
4,1.0,None,0.632911,0.409220
5,0.1,None,0.531646,0.266015


## 4. Train the selected model and evaluate the held-out test set

The best validation setting is fitted again on the training set. The test set is then used once to estimate performance on unseen services. Accuracy shows the overall percentage of correct predictions, balanced accuracy averages recall across the three classes, and Macro-F1 gives each class equal weight.

In [5]:
# Read the winning settings from the first row of the sorted table.
best_C = float(validation_table.loc[0, 'C'])
best_class_weight = validation_table.loc[0, 'class_weight']

if pd.isna(best_class_weight):
    best_class_weight = None

final_model = LogisticRegression(
    C=best_C,
    class_weight=best_class_weight,
    max_iter=2000,
    random_state=42
)

# Fit the selected classifier and obtain both class predictions and
# probabilities. Probabilities are later converted into a 0-100 score.
final_model.fit(X_train, y_train)
test_prediction = final_model.predict(X_test)
test_probability = final_model.predict_proba(X_test)

test_accuracy = accuracy_score(y_test, test_prediction)
test_balanced_accuracy = balanced_accuracy_score(y_test, test_prediction)
test_macro_f1 = f1_score(y_test, test_prediction, average='macro')

print('Selected C:', best_C)
print('Selected class_weight:', best_class_weight)
print('Test accuracy:', round(test_accuracy, 4))
print('Test balanced accuracy:', round(test_balanced_accuracy, 4))
print('Test Macro-F1:', round(test_macro_f1, 4))
print()
print(classification_report(y_test, test_prediction, zero_division=0))

Selected C: 10.0
Selected class_weight: balanced
Test accuracy: 0.7111
Test balanced accuracy: 0.6667
Test Macro-F1: 0.7085

              precision    recall  f1-score   support

        High       1.00      0.56      0.71         9
         Low       0.71      0.78      0.74        45
      Medium       0.67      0.67      0.67        36

    accuracy                           0.71        90
   macro avg       0.79      0.67      0.71        90
weighted avg       0.72      0.71      0.71        90



## 5. Convert class probabilities into a 0–100 risk score

The classifier returns one probability for each class. These probabilities are converted into a weighted score using `20 × P(Low) + 60 × P(Medium) + 100 × P(High)`.

The weighted value is then constrained to the predicted class range: Low 0–39, Medium 40–69, and High 70–100. This prevents a result such as a `Medium` label being displayed with a `High` numerical score. The score is an application-specific indicator, not a legal probability.

In [6]:
# These anchor values define the contribution of each class probability.
risk_value = {'Low': 20, 'Medium': 60, 'High': 100}
class_risk_values = np.array([risk_value[name] for name in final_model.classes_])
raw_risk_score = np.rint(test_probability @ class_risk_values).astype(int)

# Keep the numeric score consistent with the predicted risk level.
risk_score = raw_risk_score.copy()
risk_score[test_prediction == 'Low'] = np.clip(
    risk_score[test_prediction == 'Low'], 0, 39
)
risk_score[test_prediction == 'Medium'] = np.clip(
    risk_score[test_prediction == 'Medium'], 40, 69
)
risk_score[test_prediction == 'High'] = np.clip(
    risk_score[test_prediction == 'High'], 70, 100
)

# Add predictions, confidence, scores, and class probabilities to a copy of
# the test data so every result can be inspected clause by clause.
test_results = test_data.copy()
test_results['actual_risk_level'] = y_test
test_results['predicted_risk_level'] = test_prediction
test_results['prediction_confidence'] = test_probability.max(axis=1).round(4)
test_results['raw_probability_score'] = raw_risk_score
test_results['risk_score'] = risk_score

for index, class_name in enumerate(final_model.classes_):
    test_results[f'probability_{class_name.lower()}'] = (
        test_probability[:, index].round(4)
    )

test_results[[
    'clause_text_clean', 'actual_risk_level', 'predicted_risk_level',
    'prediction_confidence', 'risk_score'
]].head(10)

,clause_text_clean,actual_risk_level,predicted_risk_level,prediction_confidence,risk_score
0,Information Associated With Your Account. Your...,Medium,Medium,0.6140,52
1,"Furthermore, Valve may amend this Agreement (i...",Medium,Medium,0.4529,50
2,Harm To WhatsApp Or Our Users.,Low,Low,0.9336,23
3,If you use Steam services (e.g. the Steam Cura...,Low,Low,0.7127,35
4,When Your Content is created with or submitted...,High,High,0.4989,70
5,"3.6 Tracking Data and Cookies We use ""Cookies""...",Medium,Medium,0.7475,53
6,Location Information. We collect and use preci...,Medium,Medium,0.7300,58
7,Steam may provide links to other third-party s...,Low,Medium,0.9020,60
8,Your Reddit account has a profile page that is...,Medium,Medium,0.8067,59
9,Deleted Information. Your undelivered messages...,Medium,Low,0.3593,39


## 6. Save the reusable model and evaluation outputs

The vocabulary, TF-IDF transformer, trained classifier, and scoring information are saved together in one `.pkl` file using Python's built-in `pickle` module. Saving the preprocessing objects with the classifier ensures that future clauses receive exactly the same text transformation used during training.

The notebook also exports clause-level test predictions, the validation comparison table, and a JSON file containing the main test metrics.

In [7]:
# Package every fitted component required for future predictions.
model_bundle = {
    'count_vectorizer': count_vectorizer,
    'tfidf_transformer': tfidf_transformer,
    'model': final_model,
    'text_column': 'clause_text_clean',
    'class_names': ['Low', 'Medium', 'High'],
    'risk_value': risk_value
}

# pickle is used here instead of joblib, as requested.
with (OUTPUT_DIR / 'risk_classification_model.pkl').open('wb') as model_file:
    pickle.dump(model_bundle, model_file)

# Export human-readable CSV files for inspection and team reporting.
test_results.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
validation_table.to_csv(
    OUTPUT_DIR / 'validation_model_comparison.csv', index=False
)

# Store the selected settings, overall metrics, per-class results, and the
# main limitation in a machine-readable summary.
metrics = {
    'selected_C': best_C,
    'selected_class_weight': best_class_weight,
    'test_accuracy': float(test_accuracy),
    'test_balanced_accuracy': float(test_balanced_accuracy),
    'test_macro_f1': float(test_macro_f1),
    'classification_report': classification_report(
        y_test, test_prediction, output_dict=True, zero_division=0
    ),
    'important_limitation': (
        'The OTA labels are AI-assisted weak labels. The model measures how '
        'well it reproduces those labels, not verified legal correctness.'
    )
}

(OUTPUT_DIR / 'test_metrics.json').write_text(
    json.dumps(metrics, indent=2), encoding='utf-8'
)

print('Saved model:', OUTPUT_DIR / 'risk_classification_model.pkl')

Saved model: outputs/ota_machine_learning_risk_model/risk_classification_model.pkl


## 7. Limitations and appropriate interpretation

The initial OTA labels were generated automatically from predefined risk rules rather than being fully reviewed by legal experts. Consequently, the evaluation mainly measures how consistently Logistic Regression can reproduce the current labelling scheme. It should not be interpreted as verified legal-risk accuracy.

The High-risk class also contains substantially fewer examples, so its results are less stable than the results for Low and Medium. Future iterations should prioritise manual review of Medium- and High-risk clauses, refine the labelling guidelines, and evaluate the model on a larger independently reviewed test set.